# **Pydantic** in a nutshell (data validation)

Creating an object with Pydantic

In [ ]:
from pydantic import BaseModel
from typing import Optional     # for optional fields

# -------------------------------
# Possible questions that can occur:
# Is price a number?
# Is volume an integer?
# Is a field missing
# Is product valid?
# -------------------------------
# You could check everything manually or use PYDANTIC

class Trade(BaseModel):
    price: float
    volume: int
    currency: str
    product: str

# This is the best case, when everything is fine
trade = Trade(
    price=53.20,
    volume=100,
    currency="EUR",
    product="DE_LU_BASE"
)
print(trade)

# But even with 'messy' data Pydantic automatically checks for transformations 
trade_messy = Trade(
    price="53.20",
    volume="100",
    currency="EUR",
    product="DE_LU_BASE"
)
print(trade_messy)

# ...or gives errormessage (ValidationError)
# ValidationError: 1 validation error for Trade
# price
#   Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='abc', input_type=str]
#     For further information visit https://errors.pydantic.dev/2.13/v/float_parsing
# trade_wrong_type = Trade(
#     price="abc",
#     volume=100,
#     currency="EUR",
#     product="DE_LU_BASE"
# )
# print(trade_wrong_type)
# Same for missing fields
# ValidationError: 1 validation error for Trade
# product
#   Field required [type=missing, input_value={'price': '53.20', 'volume': 100}, input_type=dict]
#     For further information visit https://errors.pydantic.dev/2.13/v/missing
# trade_missing_type = Trade(
#     price="53.20",
#     volume=100,
#     currency="EUR"
# )
# print(trade_missing_type)

# It's possible to set standard values for your fields 
class Trade(BaseModel):
    price: float
    volume: int
    currency: str = "USD"
    product: str
trade_no_currency = Trade(
    price="53.20",
    volume=100,
    product="DE_LU_BASE"
)
print(trade_no_currency)

# ... or make it optional
class Trade(BaseModel):
    price: float
    volume: int
    currency: str | None # currency field required but None value may be provided 
    product: str
    market: Optional[str] = None

trade_opt_market = Trade(
    price="53.20",
    volume=100,
    product="DE_LU_BASE",
    currency=None,
    #market="EEX"
)
print(trade_opt_market)

Get data (from API) and validate

In [ ]:
from datetime import datetime
from pydantic import field_validator

class Trade(BaseModel):
    price: float
    volume: int
    currency: str 
    product: str
    market: Optional[str] = None
    timestamp: datetime

    # You can create your own custom validation
    @field_validator("volume")
    def validate_volume(cls, value):
        if value <= 0:
            raise ValueError(f"Volume must be non-negative: {value}")
        return value

# Example data
trade_data = {
    "price": "53.20",
    "volume": "100",
    "currency": "EUR",
    "product": "DE_LU_BASE",
    "market" : "EEX",
    "timestamp" : "2026-07-29T13:00:00"
}

trade = Trade(**trade_data)
# Trade(**trade_data) is the same as Trade(price="53.20", volume="100", currency="EUR", product="DE_LU_BASE", market="EEX", timestamp="2026-07-29T13:00:00")
print(trade.price)
print(trade.volume)
print(trade.currency)
print(trade.product)
print(trade.timestamp)

Serialization (save your model)

In [53]:
# 1 - dict made up of the associated Python objects
trade.model_dump()      # prints model 
a = trade.model_dump()  # saves model in variable a

# 2 - dict made up only of "jsonable" types
trade.model_dump(mode='json')      # prints model 
b = trade.model_dump(mode='json')  # saves model in variable b and datetime(2026,7,29,12,30) -> "2026-07-29T12:30:00"

# 3 - JSON string
trade.model_dump_json()      # prints model 
c = trade.model_dump_json()  # saves model in variable c and datetime(2026,7,29,12,30) -> "2026-07-29T12:30:00"

# Bonus - if required you can exclude some fields  
trade.model_dump(exclude={"market", "currency"})      # prints model
d = trade.model_dump(exclude={"market", "currency"})  # saves model in variable d but without market and currency 